# Exploratory Data Analysis — Edge AI Acoustic Border Intrusion Detection Dataset

**Paper Title (Draft):** *Edge AI-Based Acoustic Footstep Detection for Border Surveillance Using TinyML and LoRa*

---

### Notebook Objectives
This EDA notebook systematically characterises the `BorderIntrusionDetection-Data` acoustic corpus
prior to TinyML model development. The analysis covers:

| Section | Focus |
|---------|-------|
| §1 | Dataset inventory & file integrity |
| §2 | Class distribution & duration statistics |
| §3 | Waveform morphology per class |
| §4 | Spectral analysis (STFT, Mel, CQT) |
| §5 | MFCC feature characterisation |
| §6 | Handcrafted feature extraction & statistics |
| §7 | Inter-class feature separability (PCA, t-SNE) |
| §8 | Correlation & redundancy analysis |
| §9 | Dataset quality & edge case detection |

**Dataset Path:** `/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset`

In [ ]:
import os, glob, warnings, itertools
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import librosa
import librosa.display
import soundfile as sf

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from scipy.stats import kurtosis, skew
from scipy.signal import welch

warnings.filterwarnings("ignore")

# ── Research Plot Style ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.labelsize":    13,
    "axes.titlesize":    13,
    "axes.linewidth":    1.2,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "xtick.major.size":  6,
    "ytick.major.size":  6,
    "xtick.minor.size":  3,
    "ytick.minor.size":  3,
    "legend.fontsize":   11,
    "legend.frameon":    True,
    "legend.edgecolor":  "0.4",
    "grid.linestyle":    ":",
    "grid.linewidth":    0.7,
    "grid.alpha":        0.85,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

CLASS_COLORS = {
    "footsteps": "#2166ac",
    "noise":     "#d6604d",
    "balastic":  "#4dac26",
}
CLASS_MARKERS = {"footsteps": "o", "noise": "s", "balastic": "^"}

DATASET_PATH = Path("/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset")
SR_TARGET    = 16000   # resample target for TinyML pipeline
N_MFCC       = 13
N_FFT        = 512
HOP_LENGTH   = 256
N_MELS       = 40

os.makedirs("figures", exist_ok=True)
print("Classes found:", [p.name for p in sorted(DATASET_PATH.iterdir()) if p.is_dir()])